<a href="https://colab.research.google.com/github/viviantothewu-web/trashnet-waste-classification/blob/main/02_Shared_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shared PyTorch Training Pipeline

This notebook defines the common data pipeline that will be used to train and evaluate all four CNN architectures:

- Custom CNN
- VGG16
- ResNet50
- MobileNetV2

The goal of the shared pipeline is to ensure that differences in model performance are caused by the CNN architectures rather than differences in dataset splitting, image processing, or data loading.

## Team Instructions

All team members should use this same pipeline.

Each person is assigned a different **training seed**:

- **Vivian:** `TRAINING_SEED = 0`
- **Joshua:** `TRAINING_SEED = 1`
- **Devan:** `TRAINING_SEED = 2`

The following settings should **not be changed**
(If you guys spot anything let me know! then we can change it all together):

- `RANDOM_STATE = 42`
- Image size = `224 × 224`
- Batch size = `32`
- Class order
- Train/validation/test split
- Training augmentation
- Validation/test transformations

The file `trashnet_split.csv` contains the fixed 70/15/15 dataset split created in Notebook 01. **Do not generate a new train/test split in this notebook.**

When using Google Colab, select a GPU runtime before running the notebook.

**Using PYTORCH + GPU**

(I went into Runtime-> change runtime type -> GPU in case you guys need to follow)

In [2]:
#SELECT GPU
#Check that PyTorch can access the GPU
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
#select the device pytorch will use
# Use the GPU if CUDA is available.
# If a GPU is not available, automatically fall back to the CPU.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


**Defining the experiment setup**

- These settings should remain consistent across all four CNN architectures so that the models can be compared fairly.
- original trashnet image is 512 × 384, resizing to 224 x 224 because it is a standard input size for VGG16, ResNet50, and MobileNetV2. Using the same image size also ensures a fair comparison across all four models while keeping training computationally efficient.

In [4]:
# Seed we used to create the fixed train/validation/test split.
RANDOM_STATE = 42

# Training seed assigned to each team member:
# Vivian = 0
# Joshua = 1
# Devan = 2
TRAINING_SEED = 0

# All images will be resized to 224 x 224 pixels.
# This size works with the Custom CNN, VGG16, ResNet50, and MobileNetV2.
IMG_SIZE = 224

# Number of images processed together during training
# (let me know if you guys want ot change the batch size!)
BATCH_SIZE = 32

# TrashNet's six waste categories.
# Keep this order fixed because the position determines the numeric class label.
CLASS_NAMES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash"
]

NUM_CLASSES = len(CLASS_NAMES)

# Display the experiment configuration.
print("Random State:", RANDOM_STATE)
print("Training seed:", TRAINING_SEED)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

Random State: 42
Training seed: 0
Image size: 224
Batch size: 32
Number of classes: 6
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


now we want to use our assigned TRAINING_SEED = 0 wherever randomness is involved

In [6]:
# SET RANDOM SEEDS
import numpy as np
import random

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAINING_SEED)

print("Training seed set to:", TRAINING_SEED)

Training seed set to: 0


In [8]:
# load trashnet
# Colab storage is temporary, so download TrashNet if it is not
# already available in the current runtime.
from pathlib import Path
if not Path("trashnet").exists():
    !git clone https://github.com/garythung/trashnet.git

print("TrashNet repository available:", Path("trashnet").exists())

Cloning into 'trashnet'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 45 (delta 12), reused 6 (delta 6), pack-reused 27 (from 1)
Receiving objects: 100% (45/45), 40.64 MiB | 32.18 MiB/s, done.
Resolving deltas: 100% (14/14), done.
TrashNet repository available: True


In [9]:
# get the images
# The TrashNet images are stored in dataset-resized.zip.
# Extract them only if the dataset folder does not already exist.

DATASET_PATH = Path("trashnet/data/dataset-resized")

if not DATASET_PATH.exists():
    !unzip -q trashnet/data/dataset-resized.zip -d trashnet/data/

print("Dataset available:", DATASET_PATH.exists())
print("Dataset path:", DATASET_PATH)

Dataset available: True
Dataset path: trashnet/data/dataset-resized


In [10]:
# TrashNet dataset should contain 2,527 images across the six waste classes, check

total_images = 0

for class_name in CLASS_NAMES:
    class_path = DATASET_PATH / class_name
    image_count = len(list(class_path.glob("*.jpg")))

    print(f"{class_name}: {image_count}")
    total_images += image_count

print("\nTotal images:", total_images)

cardboard: 403
glass: 501
metal: 410
paper: 594
plastic: 482
trash: 137

Total images: 2527


already did in EDA but thats in a separate notebook so confirm again

**Now load the fixed train/validation/test split**

In [11]:
# use the csv created in Notebook 01 EDA!!

from google.colab import files

uploaded = files.upload()

Saving trashnet_split.csv to trashnet_split.csv


In [13]:
# double checking all 2,527 image are there
import pandas as pd
split_df = pd.read_csv("trashnet_split.csv")

print("Total images in split:", len(split_df))
print("\nImages per split:")
print(split_df["split"].value_counts())

split_df.head()

Total images in split: 2527

Images per split:
split
train         1768
test           380
validation     379
Name: count, dtype: int64


,filepath,class,split
0,trashnet/data/dataset-resized/plastic/plastic3...,plastic,train
1,trashnet/data/dataset-resized/cardboard/cardbo...,cardboard,train
2,trashnet/data/dataset-resized/cardboard/cardbo...,cardboard,train
3,trashnet/data/dataset-resized/cardboard/cardbo...,cardboard,train
4,trashnet/data/dataset-resized/metal/metal171.jpg,metal,train


**Nobody should call train_test_split() again during model training!!**

next, we need a consistent mapping between the six class names and numeric labels

**Mapping stays the SAME across entire projecr!! Otherwise, one model could treat 4 as plastic while another treats 4 as paper**

In [15]:
# Convert each waste class into a numeric label for PyTorch.
# This mapping must remain the same for every model

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(CLASS_NAMES)
}

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

print("Class to index:")
print(class_to_idx)

print("\nIndex to class:")
print(idx_to_class)

Class to index:
{'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}

Index to class:
{0: 'cardboard', 1: 'glass', 2: 'metal', 3: 'paper', 4: 'plastic', 5: 'trash'}


- now we take the 512x384 image and resize to 224x224
- then we convert to tensor
- we will have: 3 x224 x 224

In [17]:

# Resize each image to the shared 224x224 input size and convert it
# into a PyTorch tensor.
from torchvision import transforms
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

print(eval_transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)


- Trashnet data is small. Training images were augmented using random horizontal flips and small rotations to reduce overfitting and introduce limited visual variation. Validation and test images were not augmented.

In [18]:
# Apply small random changes to training images to improve generalization.
# These augmentations are applied only during training.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Randomly flip some images horizontally.
    transforms.RandomHorizontalFlip(p=0.5),

    # Randomly rotate images by up to 10 degrees.
    transforms.RandomRotation(10),

    # Convert the image into a PyTorch tensor.
    transforms.ToTensor()
])

print(train_transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
)


In [20]:
# This class connects the file paths and class labels from our fixed
# dataset split to the actual TrashNet image files.
from torch.utils.data import Dataset
class TrashNetDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        # Store the dataframe containing file paths and class labels.
        self.dataframe = dataframe.reset_index(drop=True)

        # Store the image transformations to apply.
        self.transform = transform

    def __len__(self):
        # Return the number of images in the dataset.
        return len(self.dataframe)

    def __getitem__(self, index):
        # Get one row from the dataframe.
        row = self.dataframe.iloc[index]

        # Get the image location and class name.
        image_path = row["filepath"]
        class_name = row["class"]

        # Open the image and make sure it is RGB.
        image = Image.open(image_path).convert("RGB")

        # Convert the class name into its numeric label.
        label = class_to_idx[class_name]

        # Apply the appropriate image transformations.
        if self.transform:
            image = self.transform(image)

        return image, label

The custom Dataset connects our fixed CSV split to the actual TrashNet images. It tells PyTorch how to load each image, apply the correct transformations, and convert its class name into a numeric label for training.

For each image, it:

1. Finds the image using its stored filepath.
2. Opens it as an RGB image.
3. Converts its class name into the fixed numeric label.
4. Applies the appropriate image transformations.
5. Returns the processed image and its correct label to PyTorch.

**now we create the 3 pytorch dataset**

- train_df= training images → WITH augmentation

- val_df = validation images → NO augmentation

- test_df = test images → NO augmentation

In [21]:
train_dataset = TrashNetDataset(
    train_df,
    transform=train_transform
)

val_dataset = TrashNetDataset(
    val_df,
    transform=eval_transform
)

test_dataset = TrashNetDataset(
    test_df,
    transform=eval_transform
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Training dataset: 1768
Validation dataset: 379
Test dataset: 380


In [23]:
#test if custom Dataset correctly loads an image and its label
from PIL import Image
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Numeric label:", label)
print("Class label:", idx_to_class[label])

Image shape: torch.Size([3, 224, 224])
Numeric label: 4
Class label: plastic


**Create Dataloaders**

In [24]:
# batches of 32 for training and evaluation
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 56
Validation batches: 12
Test batches: 12


- I did not shuffle the validation and test since the model isn't learning, so there's no benefit to shuffling. Keeping them in a fixed order also makes later predictions and error analysis easier to match back to the original images.

In [25]:
#check 1 batch

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


**now we move a batch to GPU**

In [26]:
# DataLoader batches initially live in CPU memory.
# Move the images and labels to device before trainng

images = images.to(device)
labels = labels.to(device)

print("Images device:", images.device)
print("Labels device:", labels.device)

Images device: cuda:0
Labels device: cuda:0


**DURING TRAINING**

each batch will:
- DataLoader -> CPU memory -> .to(device) -> GPU -> CNN

## Team Checklist

Before training a model, confirm:

- [ ] Google Colab GPU is enabled.
- [ ] `CUDA available` is `True`.
- [ ] The TrashNet image count is `2527`.
- [ ] `trashnet_split.csv` contains `2527` rows.
- [ ] The fixed train/validation/test split loads successfully.
- [ ] The class mapping has not been changed.
- [ ] Image size is `224 × 224`.
- [ ] Batch size is `32`.
- [ ] Your assigned `TRAINING_SEED` is correct.
- [ ] A training batch has shape `[32, 3, 224, 224]`.
- [ ] Images and labels successfully move to `cuda:0`.

**keep your assigned training seed**